In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from google.colab import drive


# 1. SETUP & EXTRACTION
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/DR_Project"
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, "DR_Aptos_Only_224")
TEMP_EXTRACT_DIR = "/content/temp_raw_data_aptos"

APTOS_ZIP = os.path.join(DRIVE_ROOT, "aptos_2019/aptos2019-blindness-detection.zip")
IMAGE_DEST_DIR = os.path.join(OUTPUT_ROOT, "images")

os.makedirs(IMAGE_DEST_DIR, exist_ok=True)
os.makedirs(TEMP_EXTRACT_DIR, exist_ok=True)


!apt-get install -y p7zip-full

def extract_with_7z(archive_path, output_dir):
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    print(f"Extracting {archive_path}...")
    subprocess.run(['7z', 'x', archive_path, f'-o{output_dir}', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

aptos_raw = os.path.join(TEMP_EXTRACT_DIR, "aptos")
extract_with_7z(APTOS_ZIP, aptos_raw)
print("Initial extraction complete.")

# 2. PATH MAPPING & STANDARDIZATION
def get_image_map(directory):
    img_map = {}
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                name_no_ext = os.path.splitext(file)[0]
                img_map[name_no_ext] = os.path.join(root, file)
    return img_map

aptos_img_map = get_image_map(aptos_raw)

aptos_csv_path = None
for root, _, files in os.walk(aptos_raw):
    if 'train.csv' in files:
        aptos_csv_path = os.path.join(root, 'train.csv')
        break

df_aptos_raw = pd.read_csv(aptos_csv_path)

aptos_records = []
for _, row in df_aptos_raw.iterrows():
    img_id = str(row['id_code'])
    if img_id in aptos_img_map:
        aptos_records.append({
            'img_id': img_id,
            'img_path': aptos_img_map[img_id],
            'label': int(row['diagnosis']),
            'source_dataset': 'aptos'
        })

df_raw_metadata = pd.DataFrame(aptos_records)
print(f"Loaded {len(df_raw_metadata)} intact APTOS records.")

# 3. PREPROCESSING (CROP, PAD, RESIZE 224)
def get_cropped_padded_base(image_path):
    """Handles loading, thresholding, cropping, and padding BEFORE resizing."""
    img = cv2.imread(image_path)
    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if mask.sum() == 0:
        cropped = img
    else:
        coords = np.argwhere(mask > 0)
        x0, y0 = coords.min(axis=0)
        x1, y1 = coords.max(axis=0) + 1
        cropped = img[x0:x1, y0:y1]

    h, w = cropped.shape[:2]
    diff = abs(h - w)
    if h < w:
        padded = cv2.copyMakeBorder(cropped, diff//2, diff-(diff//2), 0, 0, cv2.BORDER_CONSTANT, value=[0,0,0])
    else:
        padded = cv2.copyMakeBorder(cropped, 0, 0, diff//2, diff-(diff//2), cv2.BORDER_CONSTANT, value=[0,0,0])

    return padded

SEVERITY_MAP = {0: "No", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}
output_records = []
TARGET_RES = 224

print(f"Extracting ROIs and generating {TARGET_RES}x{TARGET_RES} images...")
for _, row in df_raw_metadata.iterrows():
    src_path = row['img_path']
    label = row['label']
    safe_name = f"{row['img_id']}.png" # Standardizing output extension to PNG

    base_img = get_cropped_padded_base(src_path)

    if base_img is not None:
        h_current = base_img.shape[0]
        interpolation = cv2.INTER_AREA if TARGET_RES < h_current else cv2.INTER_CUBIC
        resized_rgb = cv2.resize(base_img, (TARGET_RES, TARGET_RES), interpolation=interpolation)
        resized_bgr = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2BGR)

        dest_path = os.path.join(IMAGE_DEST_DIR, safe_name)
        cv2.imwrite(dest_path, resized_bgr)

        severity_text = SEVERITY_MAP[label]
        prompt = f"A fundus photograph showing {severity_text} diabetic retinopathy."

        output_records.append({
            'image_file': safe_name,
            'label': label,
            'source_dataset': row['source_dataset'],
            'vlm_prompt': prompt
        })

df_master = pd.DataFrame(output_records)

# 4. STRATIFIED SPLITTING (80/10/10)
print("\nCalculating stratified 80/10/10 splits...")

# Split out 20% for val/test combined, keeping class distribution identical
df_train, df_val_test = train_test_split(
    df_master,
    test_size=0.20,
    stratify=df_master['label'],
    random_state=42
)

# Split the remaining 20% equally into 10% val and 10% test
df_val, df_test = train_test_split(
    df_val_test,
    test_size=0.50,
    stratify=df_val_test['label'],
    random_state=42
)

# Assign split labels
df_train = df_train.copy()
df_val = df_val.copy()
df_test = df_test.copy()

df_train['split'] = 'train'
df_val['split'] = 'val'
df_test['split'] = 'test'

# Recombine and sort for a clean final CSV
df_final_split = pd.concat([df_train, df_val, df_test]).sort_index()

csv_out_path = os.path.join(OUTPUT_ROOT, "aptos_dataset_80_10_10.csv")
df_final_split.to_csv(csv_out_path, index=False)

# 5. FINAL STATS CHECK
print("PIPELINE COMPLETE! HERE ARE YOUR STATS:")
print(f"Total Images Processed: {len(df_final_split)}")
print(f"\nOverall Label Distribution:\n{df_final_split['label'].value_counts().sort_index()}")

print("\nSplit Sizes:")
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

print("\nTraining Set Class Proportions:")
print(df_train['label'].value_counts(normalize=True).mul(100).round(2))

print(f"\nFiles and CSV successfully saved to: {OUTPUT_ROOT}")

Mounted at /content/drive
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Extracting /content/drive/MyDrive/DR_Project/aptos_2019/aptos2019-blindness-detection.zip...
Initial extraction complete.
Loaded 3662 intact APTOS records.
Extracting ROIs and generating 224x224 images...

Calculating stratified 80/10/10 splits...

✅ PIPELINE COMPLETE! HERE ARE YOUR STATS:
Total Images Processed: 3662

Overall Label Distribution:
label
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Split Sizes:
Train: 2929 | Val: 366 | Test: 367

Training Set Class Proportions:
label
0    49.30
2    27.28
1    10.11
4     8.06
3     5.26
Name: proportion, dtype: float64

Files and CSV successfully saved to: /content/drive/MyDrive/DR_Project/DR_Aptos_Only_224


CLAHE 512

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from google.colab import drive

# ==========================================
# 1. SETUP & EXTRACTION
# ==========================================
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/DR_Project"
# CHANGED: Updated folder name to reflect 512 and CLAHE
OUTPUT_ROOT = os.path.join(DRIVE_ROOT, "DR_Aptos_Only_512_CLAHE")
TEMP_EXTRACT_DIR = "/content/temp_raw_data_aptos"

APTOS_ZIP = os.path.join(DRIVE_ROOT, "aptos_2019/aptos2019-blindness-detection.zip")
IMAGE_DEST_DIR = os.path.join(OUTPUT_ROOT, "images")

os.makedirs(IMAGE_DEST_DIR, exist_ok=True)
os.makedirs(TEMP_EXTRACT_DIR, exist_ok=True)

# Install p7zip for robust extraction
!apt-get install -y p7zip-full

def extract_with_7z(archive_path, output_dir):
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    print(f"Extracting {archive_path}...")
    subprocess.run(['7z', 'x', archive_path, f'-o{output_dir}', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

aptos_raw = os.path.join(TEMP_EXTRACT_DIR, "aptos")
extract_with_7z(APTOS_ZIP, aptos_raw)
print("Initial extraction complete.")

# ==========================================
# 2. PATH MAPPING & STANDARDIZATION
# ==========================================
def get_image_map(directory):
    img_map = {}
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                name_no_ext = os.path.splitext(file)[0]
                img_map[name_no_ext] = os.path.join(root, file)
    return img_map

aptos_img_map = get_image_map(aptos_raw)

aptos_csv_path = None
for root, _, files in os.walk(aptos_raw):
    if 'train.csv' in files:
        aptos_csv_path = os.path.join(root, 'train.csv')
        break

df_aptos_raw = pd.read_csv(aptos_csv_path)

aptos_records = []
for _, row in df_aptos_raw.iterrows():
    img_id = str(row['id_code'])
    if img_id in aptos_img_map:
        aptos_records.append({
            'img_id': img_id,
            'img_path': aptos_img_map[img_id],
            'label': int(row['diagnosis']),
            'source_dataset': 'aptos'
        })

df_raw_metadata = pd.DataFrame(aptos_records)
print(f"Loaded {len(df_raw_metadata)} intact APTOS records.")

# ==========================================
# 3. PREPROCESSING (CROP, PAD, RESIZE 512, CLAHE)
# ==========================================
def get_cropped_padded_base(image_path):
    """Handles loading, thresholding, cropping, and padding BEFORE resizing."""
    img = cv2.imread(image_path)
    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if mask.sum() == 0:
        cropped = img
    else:
        coords = np.argwhere(mask > 0)
        x0, y0 = coords.min(axis=0)
        x1, y1 = coords.max(axis=0) + 1
        cropped = img[x0:x1, y0:y1]

    h, w = cropped.shape[:2]
    diff = abs(h - w)
    if h < w:
        padded = cv2.copyMakeBorder(cropped, diff//2, diff-(diff//2), 0, 0, cv2.BORDER_CONSTANT, value=[0,0,0])
    else:
        padded = cv2.copyMakeBorder(cropped, 0, 0, diff//2, diff-(diff//2), cv2.BORDER_CONSTANT, value=[0,0,0])

    return padded

# ADDED: CLAHE Function
def apply_clahe_bgr(img_bgr):
    """Applies CLAHE to the Lightness channel of LAB color space."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

SEVERITY_MAP = {0: "No", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}
output_records = []
TARGET_RES = 512 # CHANGED to 512

print(f"Extracting ROIs, applying CLAHE, and generating {TARGET_RES}x{TARGET_RES} images...")
for _, row in df_raw_metadata.iterrows():
    src_path = row['img_path']
    label = row['label']
    safe_name = f"{row['img_id']}.png" # Standardizing output extension to PNG

    base_img = get_cropped_padded_base(src_path)

    if base_img is not None:
        h_current = base_img.shape[0]
        interpolation = cv2.INTER_AREA if TARGET_RES < h_current else cv2.INTER_CUBIC
        resized_rgb = cv2.resize(base_img, (TARGET_RES, TARGET_RES), interpolation=interpolation)
        resized_bgr = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2BGR)

        # ADDED: Apply CLAHE after resizing
        clahe_bgr = apply_clahe_bgr(resized_bgr)

        dest_path = os.path.join(IMAGE_DEST_DIR, safe_name)
        cv2.imwrite(dest_path, clahe_bgr) # Save the CLAHE version

        severity_text = SEVERITY_MAP[label]
        prompt = f"A fundus photograph showing {severity_text} diabetic retinopathy."

        output_records.append({
            'image_file': safe_name,
            'label': label,
            'source_dataset': row['source_dataset'],
            'vlm_prompt': prompt
        })

df_master = pd.DataFrame(output_records)

# ==========================================
# 4. STRATIFIED SPLITTING (80/10/10)
# ==========================================
print("\nCalculating stratified 80/10/10 splits...")

# Split out 20% for val/test combined, keeping class distribution identical
df_train, df_val_test = train_test_split(
    df_master,
    test_size=0.20,
    stratify=df_master['label'],
    random_state=42
)

# Split the remaining 20% equally into 10% val and 10% test
df_val, df_test = train_test_split(
    df_val_test,
    test_size=0.50,
    stratify=df_val_test['label'],
    random_state=42
)

# Assign split labels
df_train = df_train.copy()
df_val = df_val.copy()
df_test = df_test.copy()

df_train['split'] = 'train'
df_val['split'] = 'val'
df_test['split'] = 'test'

# Recombine and sort for a clean final CSV
df_final_split = pd.concat([df_train, df_val, df_test]).sort_index()

csv_out_path = os.path.join(OUTPUT_ROOT, "aptos_dataset_80_10_10.csv")
df_final_split.to_csv(csv_out_path, index=False)

# ==========================================
# 5. FINAL STATS CHECK
# ==========================================
print("\n=========================================")
print("✅ PIPELINE COMPLETE! HERE ARE YOUR STATS:")
print("=========================================")
print(f"Total Images Processed: {len(df_final_split)}")
print(f"\nOverall Label Distribution:\n{df_final_split['label'].value_counts().sort_index()}")

print("\nSplit Sizes:")
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

print("\nTraining Set Class Proportions:")
print(df_train['label'].value_counts(normalize=True).mul(100).round(2))

print(f"\nFiles and CSV successfully saved to: {OUTPUT_ROOT}")

Mounted at /content/drive
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Extracting /content/drive/MyDrive/DR_Project/aptos_2019/aptos2019-blindness-detection.zip...
Initial extraction complete.
Loaded 3662 intact APTOS records.
Extracting ROIs, applying CLAHE, and generating 512x512 images...

Calculating stratified 80/10/10 splits...

✅ PIPELINE COMPLETE! HERE ARE YOUR STATS:
Total Images Processed: 3662

Overall Label Distribution:
label
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Split Sizes:
Train: 2929 | Val: 366 | Test: 367

Training Set Class Proportions:
label
0    49.30
2    27.28
1    10.11
4     8.06
3     5.26
Name: proportion, dtype: float64

Files and CSV successfully saved to: /content/drive/MyDrive/DR_Project/DR_Aptos_Only_512_CLAHE
